# 01 — Basic Damage Simulation

Run a single scenario (Warrior vs NPC) and inspect the damage distribution.

In [ ]:
from pathlib import Path

from omega.model.constants import SKILLID_SWORDSMANSHIP, SKILLID_TACTICS, SKILLID_ANATOMY
from omega.shard import ShardData
from omega.simulation import (
    ArmorSpec, CombatantSpec, Scenario, WeaponSpec, run_scenario,
)
from omega.reporting.tables import summary_table, format_table_html
from omega.reporting.plots import damage_histogram, damage_breakdown

SHARD_ROOT = Path("..") / "submodules" / "zuluhotel_omega_2.5"
shard = ShardData.from_path(SHARD_ROOT)
parse_results = shard.parse_combat_scripts()

In [ ]:
scenario = Scenario(
    attacker=CombatantSpec(
        name="Warrior",
        skills={SKILLID_SWORDSMANSHIP: 100, SKILLID_TACTICS: 100, SKILLID_ANATOMY: 100},
        str_=100, dex_=100, int_=25,
        class_levels={"IsWarrior": 5},
        weapon=WeaponSpec(name="Broadsword", damage="3d6+2", attribute=SKILLID_SWORDSMANSHIP),
    ),
    defender=CombatantSpec(
        name="Target Dummy",
        is_npc=True,
        str_=50, dex_=50, int_=50,
        hp=500,
        armor=ArmorSpec(name="Plate", ar=30),
    ),
    iterations=200,
    base_seed=42,
)

result = run_scenario(
    scenario,
    parse_results=parse_results,
    config_resolver=shard.resolve_config_path,
    em_modules_dir=shard.root / "scripts" / "modules",
)
print(f"Successes: {result.success_count}/{result.iteration_count}")
print(f"Mean damage: {result.damage_stats.mean:.2f}")
print(f"Range: {result.damage_stats.min:.0f} – {result.damage_stats.max:.0f}")

In [ ]:
# Damage distribution histogram
damage_histogram(result, title="Warrior vs NPC — 200 hits")

In [ ]:
# Base / absorbed / final breakdown
damage_breakdown(result, title="Damage Breakdown")

In [ ]:
# HTML summary table
from IPython.display import HTML
from omega.simulation.stats import SimulationResult

rows = summary_table(SimulationResult(cells=[result]), stats=["mean", "median", "min", "max", "p5", "p95", "hit_rate"])
HTML(format_table_html(rows))